# 08 — Métricas RAGAS (juez vía Ollama + cálculo determinista)

Dos capas: (1) las métricas **deterministas** estilo RAGAS (`rag_baseline/ragas/compute_metrics.py`: context precision/recall, faithfulness por chequeo de claims, relevancy por coseno) y (2) el run **real con la librería `ragas` 0.4.4**, ahora con el juez vía **ChatOllama** (`JUDGE_OLLAMA_MODEL`, por defecto `qwen2.5:7b` — el juez 1.5B en CPU se atoró incluso con timeout de 1800 s por job; en la T4 vía Ollama debería completar). Si el run real no completa, las deterministas siguen siendo las cifras citables, y el notebook lo dice.

**Prerrequisitos:** notebooks 05 y 06 (trazas y retrievals de las tres variantes RAG). El juez no evalúa al grafo ni a MEMO: solo a las tres variantes RAG, como en local.

In [ ]:
%%bash
nvidia-smi
python3 -c "import torch; print('torch', torch.__version__, '| CUDA disponible:', torch.cuda.is_available())" 2>/dev/null || echo "torch aun no instalado (se instala abajo)"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive montado en /content/drive")

In [ ]:
# --- Parametros (editalos aqui arriba) ---
DRIVE_ROOT = "/content/drive/MyDrive/femsa-rag"   # carpeta en Drive donde vive todo
REPO_URL = "https://github.com/czammar/femsa-rag-graph-memo"
REPO = DRIVE_ROOT + "/femsa-rag-graph-memo"

# --- Parametros ---
JUDGE_OLLAMA_MODEL = "qwen2.5:7b"   # juez: 7b (default, mejor juez) o qwen2.5:1.5b (paridad CPU)
EVAL_SET = "100"   # set de evaluacion: "100" (default, 40/30/30) o "12" (original)
EVAL_FILE = "eval_questions_100.json" if EVAL_SET == "100" else "eval_questions.json"
SUF = "_100" if EVAL_SET == "100" else ""
N_Q = 100 if EVAL_SET == "100" else 12
TIME_BUDGET_MIN = 90   # tope de tiempo para el run real con ragas (muestras no hechas = skipped, no numeros)

In [ ]:
import os, subprocess

os.makedirs(DRIVE_ROOT, exist_ok=True)
# El cache de Hugging Face vive en Drive: los pesos descargados
# sobreviven al reinicio de la sesion (la VM de Colab es efimera).
os.environ["HF_HOME"] = DRIVE_ROOT + "/.cache/huggingface"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

if os.path.isdir(REPO + "/.git"):
    print("repo ya existe, actualizando...")
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=False)
else:
    print("clonando el repo...")
    subprocess.run(["git", "clone", REPO_URL, REPO], check=True)
print("repo en:", REPO)
print("HF_HOME:", os.environ["HF_HOME"])

## Datos procesados: primero en Drive, si no, súbelos

Este notebook necesita `pipeline/data/chunks.jsonl` (las 151 páginas del informe extraídas). El flujo es:

1. **Buscarlo en Drive** (ruta estándar, bajo el repo clonado).
2. **Si no está, subirlo** con el widget de Colab de la celda siguiente: el archivo es `chunks.jsonl`. Lo consigues de una corrida anterior en Drive.
3. **Solo si no lo subes**, generarlo aquí mismo con la lógica de extracción inline (descarga el PDF ~19 MB y extrae las 151 páginas, ~1–2 min). Es el último recurso, no el camino principal. La lógica es la de `pipeline/extract.py` del repo, copiada en la celda: el notebook no ejecuta scripts del repo.

`triples_seed.json`, `eval_questions.json` y `eval_questions_100.json` vienen con el clon del repo: no hace falta subirlos.

In [ ]:
import json, subprocess, sys
from pathlib import Path

CHUNKS = REPO + "/pipeline/data/chunks.jsonl"
DATA_DIR = REPO + "/pipeline/data"


def _chunks_ok(path):
    try:
        with open(path) as f:
            first = f.readline()
            n = 1 + sum(1 for _ in f)
        row = json.loads(first)
        return n >= 100 and isinstance(row.get("text"), str) and len(row["text"]) > 0
    except Exception:
        return False


if _chunks_ok(CHUNKS):
    print(f"datos procesados OK en Drive: {CHUNKS}")
else:
    print("chunks.jsonl NO está en Drive.")
    print("Súbelo con el widget de abajo (botón 'Elegir archivos').")
    print("Qué subir: chunks.jsonl — las 151 páginas del informe FEMSA 2024.")
    print("Dónde conseguirlo: de una corrida anterior en Drive.")
    print("Si no lo subes, el notebook lo genera aquí mismo con la lógica inline (último recurso).")
    from google.colab import files
    uploaded = files.upload()
    for name, data in uploaded.items():
        dest = Path(DATA_DIR) / Path(name).name
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(data)
        print(f"subido: {dest} ({len(data)} bytes)")
    if not _chunks_ok(CHUNKS) and len(uploaded) == 1:
        # El archivo subido puede venir con otro nombre: probar como chunks.jsonl.
        only = next(iter(uploaded.values()))
        Path(CHUNKS).write_bytes(only)
        print("probando el archivo subido como chunks.jsonl...")
    if _chunks_ok(CHUNKS):
        print("datos listos por subida.")
    else:
        print("Sin subida válida: último recurso, generar los chunks aquí mismo...")
        print("(lógica de extracción inline: descarga el PDF ~19 MB y extrae las 151 páginas, ~1-2 min)")
        # El PDF está cifrado con AES (contraseña vacía): pypdf necesita el
        # paquete opcional `cryptography` para leerlo, que `pip install pypdf`
        # no trae. Se verifica aquí para que el último recurso funcione.
        try:
            import pypdf
            import cryptography
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                            "pypdf>=7.1.1", "cryptography>=3.1"], check=True)
            import pypdf
            import cryptography
        import urllib.request
        from pypdf import PdfReader

        # Lógica de extracción del informe, copiada inline: este notebook no
        # ejecuta scripts del repo. Solo la edición en inglés: el sitio en
        # español bloquea las descargas directas (WAF), así que la edición en
        # inglés es la que se descarga (mismo informe, mismas cifras).
        _PDF_URL_EN = "https://annualreport.femsa.com/pdf/FEMSA_2024_Annual_Report.pdf"
        _PDF_PATH_EN = Path(DATA_DIR) / "FEMSA_2024_Annual_Report_EN.pdf"
        _SOURCE_LABEL_EN = "FEMSA 2024 Integrated Annual Report, «Evolving» (English edition, same figures)"

        def _inline_is_valid_pdf(path):
            try:
                return len(PdfReader(str(path)).pages) > 0
            except Exception:
                return False

        def _inline_download_pdf():
            if _PDF_PATH_EN.exists() and _inline_is_valid_pdf(_PDF_PATH_EN):
                print(f"PDF en caché: {_PDF_PATH_EN} ({_PDF_PATH_EN.stat().st_size / 1e6:.1f} MB)")
                return _PDF_PATH_EN
            if _PDF_PATH_EN.exists():
                print(f"Caché inválido, se vuelve a descargar: {_PDF_PATH_EN}")
                _PDF_PATH_EN.unlink()
            Path(DATA_DIR).mkdir(parents=True, exist_ok=True)
            print(f"Descargando {_PDF_URL_EN} ...")
            req = urllib.request.Request(
                _PDF_URL_EN, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=300) as resp:
                with open(_PDF_PATH_EN, "wb") as fh:
                    fh.write(resp.read())
            if not _inline_is_valid_pdf(_PDF_PATH_EN):
                _PDF_PATH_EN.unlink()
                raise RuntimeError("El archivo descargado no es un PDF válido.")
            print(f"Guardado en {_PDF_PATH_EN} ({_PDF_PATH_EN.stat().st_size / 1e6:.1f} MB)")
            return _PDF_PATH_EN

        def _inline_extract_pages(pdf_path):
            reader = PdfReader(str(pdf_path))
            chunks = []
            for i, page in enumerate(reader.pages, start=1):
                text = (page.extract_text() or "").strip()
                chunks.append({"page": i, "text": text, "chars": len(text),
                               "fuente": _SOURCE_LABEL_EN})
            return chunks

        try:
            _pdf = _inline_download_pdf()
            _chunks = _inline_extract_pages(_pdf)
            _non_empty = sum(1 for c in _chunks if c["chars"] > 0)
            print(f"Páginas: {len(_chunks)} ({_non_empty} con texto)")
            Path(CHUNKS).parent.mkdir(parents=True, exist_ok=True)
            with open(CHUNKS, "w", encoding="utf-8") as fh:
                for c in _chunks:
                    fh.write(json.dumps(c, ensure_ascii=False) + "\n")
            print(f"{len(_chunks)} páginas -> {CHUNKS}")
        except Exception as e:
            print("Generación inline falló:", e)
        assert _chunks_ok(CHUNKS), "No se pudo obtener chunks.jsonl."
    print(f"chunks.jsonl listo: {CHUNKS}")

In [ ]:
%pip install -q "torch>=2.15.0" transformers==5.24.1 sentence-transformers==6.3.0 numpy>=2.5.5 langchain-ollama langchain-huggingface==1.3.0 ragas==0.4.4

## Ollama (setup compacto e idempotente)

Este notebook usa modelos intermedios vía **Ollama** corriendo en la propia instancia de Colab (un solo servidor, cambio de modelo con un parámetro, sin cargar pesos en cada notebook). Las celdas son idempotentes: verifican si el binario existe, si el servidor ya responde y si el modelo ya está bajado antes de hacer nada. La versión completa, con la explicación y las fuentes, está en `00_ollama_setup.ipynb`.

In [ ]:
%%bash
if command -v ollama >/dev/null 2>&1; then
  echo "ollama ya instalado: $(ollama --version)"
else
  echo "instalando Ollama (primero zstd, que el instalador exige y Colab no trae)..."
  apt-get -qq update && apt-get -qq install -y zstd pciutils
  curl -fsSL https://ollama.com/install.sh | sh
  ollama --version
fi

In [ ]:
import os, subprocess, time, urllib.request

def ollama_up():
    try:
        with urllib.request.urlopen("http://localhost:11434", timeout=3) as r:
            return r.status == 200
    except Exception:
        return False

if ollama_up():
    print("servidor Ollama ya responde en http://localhost:11434")
else:
    print("arrancando `ollama serve` en segundo plano...")
    out = open("ollama.log", "w")
    err = open("ollama_error.log", "w")
    subprocess.Popen(["nohup", "ollama", "serve"], stdout=out, stderr=err,
                     preexec_fn=os.setsid)
    for _ in range(60):
        if ollama_up():
            break
        time.sleep(2)
    print("Ollama responde en localhost:11434:", ollama_up())
    if not ollama_up():
        print("Revisa ollama_error.log; el servidor no levanto.")

In [ ]:
%%bash
if ollama show "qwen2.5:7b" >/dev/null 2>&1; then
  echo "qwen2.5:7b ya descargado"
else
  echo "bajando qwen2.5:7b..."
  ollama pull "qwen2.5:7b"
fi
if ollama show "qwen2.5:1.5b" >/dev/null 2>&1; then
  echo "qwen2.5:1.5b ya descargado"
else
  echo "bajando qwen2.5:1.5b..."
  ollama pull "qwen2.5:1.5b"
fi
ollama list

In [ ]:
from langchain_ollama import ChatOllama

_llm = ChatOllama(model=JUDGE_OLLAMA_MODEL, base_url="http://localhost:11434", temperature=0)
print(_llm.invoke("Responde con una sola palabra: ¿cuánto es 2+2?").content)
print("ChatOllama OK contra", JUDGE_OLLAMA_MODEL)

## Prerrequisitos: trazas y retrievals de los notebooks 05 y 06

El cálculo determinista inline lee los nombres de CPU (`retrieval.json`, `run_trace.jsonl`, ...), pero en Colab los notebooks 05 y 06 escriben las trazas como `run_trace_colab{SUF}.jsonl` (y variantes). Esta celda verifica que existan los 6 insumos, te pide subirlos si faltan, y alinea los nombres (copia `run_trace_*_colab.jsonl` -> `run_trace_*.jsonl`) para que el cálculo inline los lea con los nombres de CPU.

In [ ]:
import json, shutil
from pathlib import Path

BASE = Path(REPO) / "rag_baseline"

# (nombre Colab, nombre que espera el calculo determinista, quién lo produce)
NEEDED = [
    (f"retrieval{SUF}.json", f"retrieval{SUF}.json", "NB05 (denso)"),
    (f"run_trace_colab{SUF}.jsonl", f"run_trace{SUF}.jsonl", "NB05 (denso)"),
    (f"retrieval_hybrid{SUF}.json", f"retrieval_hybrid{SUF}.json", "NB06 (híbrido)"),
    (f"run_trace_hybrid_colab{SUF}.jsonl", f"run_trace_hybrid{SUF}.jsonl", "NB06 (híbrido)"),
    (f"retrieval_expanded{SUF}.json", f"retrieval_expanded{SUF}.json", "NB06 (expansión)"),
    (f"run_trace_expanded_colab{SUF}.jsonl", f"run_trace_expanded{SUF}.jsonl", "NB06 (expansión)"),
]


def _ok(p):
    try:
        return p.is_file() and p.stat().st_size > 0
    except Exception:
        return False


missing = [c for c, _, _ in NEEDED if not _ok(BASE / c)]
if missing:
    print("Faltan insumos de 05/06:", missing)
    print("Súbelos con el widget de abajo (botón 'Elegir archivos').")
    print("Qué subir: los archivos de la lista (trazas run_trace_*_colab.jsonl y")
    print("retrievales retrieval*.json) tal cual salieron de los notebooks 05 y 06.")
    print("Dónde conseguirlos: de la corrida de 05/06 en Drive, o corriendo 05 y 06.")
    from google.colab import files
    uploaded = files.upload()
    for name, data in uploaded.items():
        dest = BASE / Path(name).name
        dest.write_bytes(data)
        print(f"subido: {dest} ({len(data)} bytes)")

still = [(c, src) for c, _, src in NEEDED if not _ok(BASE / c)]
assert not still, (
    "FALTAN insumos de 05/06: " + ", ".join(c for c, _ in still) + "\n"
    "Corre los notebooks 05 (denso) y 06 (híbrido + expansión) primero, "
    "o sube sus archivos (retrieval*.json y run_trace_*_colab.jsonl) a rag_baseline/."
)

# Alinear nombres: el calculo determinista espera los de CPU.
for colab_name, cpu_name, _src in NEEDED:
    if colab_name == cpu_name:
        continue
    src, dst = BASE / colab_name, BASE / cpu_name
    if _ok(src) and (not _ok(dst) or src.stat().st_mtime > dst.stat().st_mtime):
        shutil.copy(src, dst)
        print(f"alineado: {colab_name} -> {cpu_name}")
    elif _ok(dst):
        print(f"ya existe {cpu_name} (se usa ese; {colab_name} no se copia)")
print("prerrequisitos OK: trazas y retrievals de 05/06 listos.")

## 1. Métricas deterministas (siempre disponibles)

Lógica inline en las tres celdas siguientes, tomada de `rag_baseline/ragas/compute_metrics.py` y `compute_relevancy.py` (mismas definiciones, mismos outputs): precision y recall de contexto desde los traces de retrieval + ground truth auditable, faithfulness por chequeo de claims contra el contexto condensado, relevancy como proxy de coseno. Escribe `metrics_ragas{SUF}.json` igual que en local. En local: denso recall 0.47 / faithfulness 0.97, híbrido 0.53 / 0.97, expansión 1.00 / 0.98.

In [ ]:
import json, re, unicodedata
from pathlib import Path

BASE = Path(REPO) / "rag_baseline"
RAGAS_DIR = BASE / "ragas"
GT = json.load(open(RAGAS_DIR / f"ground_truth_evidence{SUF}.json"))
GT_BY_ID = {q["id"]: q for q in GT["questions"]}

VARIANTS = {
    "dense": {"retrieval": f"retrieval{SUF}.json", "trace": f"run_trace{SUF}.jsonl"},
    "hybrid": {"retrieval": f"retrieval_hybrid{SUF}.json", "trace": f"run_trace_hybrid{SUF}.jsonl"},
    "expanded": {"retrieval": f"retrieval_expanded{SUF}.json", "trace": f"run_trace_expanded{SUF}.jsonl"},
}

# Veredictos de exactitud de los resultados publicados (mismo scorer).
# Solo existen para el set de 12 (runs en CPU); con el set de 100 no hay
# run previo: los conjuntos quedan vacios y se llenan tras el run.
if EVAL_SET == "12":
    EXACTNESS = {
        "dense": {"Q6", "Q11"},
        "hybrid": {"Q1", "Q11"},
        # Q11 expanded es "correcta con alucinacion" en results_expanded.md: la
        # sustancia es correcta (Digital@FEMSA -> Spin) pero el scorer marca '328';
        # el conteo publicado es 5/12, asi que Q11 NO esta en el conjunto correcto.
        "expanded": {"Q1", "Q4", "Q6", "Q8", "Q10"},
    }
    EXACTNESS_NOTES = {"expanded": {"Q11": "correcta con alucinacion segun el scorer (artefacto: '328' = 32.8% real, p.34)"}}
else:
    EXACTNESS = {"dense": set(), "hybrid": set(), "expanded": set()}
    EXACTNESS_NOTES = {}


def norm_text(s):
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    # collapse all whitespace (PDF text has tabs/newlines inside names:
    # "Coca-Cola\tFEMSA" must match "coca-cola femsa")
    return re.sub(r"\s+", " ", s).strip().lower()


NUM_RE = re.compile(r"\d{1,3}(?:,\d{3})+(?:\.\d+)?|\d+(?:[.,]\d+)?")


def norm_number(tok):
    # '781,585' -> '781585'; '11,2' -> '11.2'; '8.6' -> '8.6'; '571' -> '571'.
    tok = tok.strip().rstrip("%")
    if "," in tok and "." not in tok:
        parts = tok.split(",")
        if len(parts) == 2 and len(parts[1]) == 1:
            return parts[0] + "." + parts[1]  # coma decimal (espanol)
        return "".join(parts)  # separador de miles
    return tok.replace(",", "")


def extract_numbers(text):
    return [norm_number(m.group(0)) for m in NUM_RE.finditer(text)]


ENTITIES = {
    # canonical -> accepted surface forms (Spanish answer vs English corpus:
    # a claim is supported if ANY form appears in the context)
    "FEMSA Forward": ["femsa forward"],
    "Digital@FEMSA": ["digital@femsa"],
    "Spin by OXXO": ["spin by oxxo"],
    "Spin Premia": ["spin premia"],
    "Proximidad Americas": ["proximidad americas", "proximity americas"],
    "Coca-Cola FEMSA": ["coca-cola femsa"],
    "Proximidad Europa": ["proximidad europa", "proximity europe"],
    "OXXO GAS": ["oxxo gas"],
    "Salud": ["salud", "health"],
    "Spin": ["spin"],
    "Mexico": ["mexico"],
    "Colombia": ["colombia"],
    "Chile": ["chile"],
    "Peru": ["peru"],
    "Argentina": ["argentina"],
    "Austria": ["austria"],
    "Brasil": ["brasil", "brazil"],
    "Costa Rica": ["costa rica"],
    "Ecuador": ["ecuador"],
    "Espana": ["espana", "spain"],
    "Guatemala": ["guatemala"],
    "Honduras": ["honduras"],
    "Paises Bajos": ["paises bajos", "netherlands"],
    "Nicaragua": ["nicaragua"],
    "Panama": ["panama"],
    "Uruguay": ["uruguay"],
    "Estados Unidos": ["estados unidos", "united states"],
}


def extract_entities(text):
    nt = norm_text(text)
    found = []
    for canonical, forms in ENTITIES.items():
        if any(f in nt for f in forms):
            found.append(canonical)
    # drop "Spin" if a longer Spin entity is present
    if any(e in found for e in ("Spin by OXXO", "Spin Premia")) and "Spin" in found:
        found.remove("Spin")
    return found


def context_numbers(context):
    body = re.sub(r"\[P[aá]gina \d+\]", " ", context)  # strip page headers
    return set(extract_numbers(body))


def context_entities(context):
    return set(extract_entities(context))


def load_variant(name):
    cfg = VARIANTS[name]
    ret = json.load(open(BASE / cfg["retrieval"]))
    pages = {q["id"]: [c["page"] for c in q["retrieved"]] for q in ret["questions"]}
    trace = [json.loads(l) for l in open(BASE / cfg["trace"])]
    answers = {t["id"]: t for t in trace}
    return pages, answers

In [ ]:
# context precision / context recall / faithfulness (determinista).
# Definiciones: precision = paginas recuperadas que son evidencia / paginas
# recuperadas (top-5); recall = items de evidencia con >=1 pagina en el top-5
# / items de evidencia (Q9 no tiene evidencia en el corpus -> recall null, no 0);
# faithfulness = claims de la respuesta apoyados por el contexto condensado /
# claims (numericos + entidades; abstencion "No lo se" -> 1.0 por convencion).
out = {"variants": {}, "method": "deterministic (see ragas/README.md)", "judge": None}
for name in VARIANTS:
    pages_by_q, answers = load_variant(name)
    per_q = {}
    for qid, gt in GT_BY_ID.items():
        retrieved = pages_by_q[qid]
        gt_pages = set()
        for item in gt["evidence"]:
            gt_pages.update(item["pages"])
        # context precision
        relevant = [p for p in retrieved if p in gt_pages]
        ctx_precision = len(relevant) / len(retrieved) if retrieved else None
        # context recall (a nivel de item de evidencia)
        n_items = len(gt["evidence"])
        if n_items == 0:
            ctx_recall = None
            recall_detail = {"covered": 0, "total": 0, "note": "no hay evidencia en el corpus (cifras calculadas, no impresas)"}
        else:
            covered = sum(1 for item in gt["evidence"] if any(p in retrieved for p in item["pages"]))
            ctx_recall = covered / n_items
            recall_detail = {"covered": covered, "total": n_items}
        # faithfulness
        ans = answers[qid]["final_answer"]
        ctx = answers[qid]["context"]
        abstain = ans.strip().rstrip(".").lower().startswith("no lo s")
        ctx_nums = context_numbers(ctx)
        ctx_ents = context_entities(ctx)
        claims = []
        if abstain:
            faithfulness = 1.0
            claims = []
        else:
            for n in extract_numbers(ans):
                claims.append({"claim": n, "type": "numeric", "supported": n in ctx_nums})
            for e in extract_entities(ans):
                claims.append({"claim": e, "type": "entity", "supported": e in ctx_ents})
            faithfulness = (sum(1 for c in claims if c["supported"]) / len(claims)) if claims else 1.0
        per_q[qid] = {
            "question": gt["question"],
            "retrieved_pages": retrieved,
            "ground_truth_pages": sorted(gt_pages),
            "context_precision": ctx_precision,
            "context_recall": ctx_recall,
            "recall_detail": recall_detail,
            "faithfulness": faithfulness,
            "n_claims": len(claims),
            "abstention": abstain,
            "claims": claims,
            "exactness_correct": qid in EXACTNESS[name],
            "exactness_note": EXACTNESS_NOTES.get(name, {}).get(qid, ""),
        }
    out["variants"][name] = {"per_question": per_q}
json.dump(out, open(RAGAS_DIR / f"metrics_ragas{SUF}.json", "w"), ensure_ascii=False, indent=1)
print("wrote metrics_ragas.json (precision/recall/faithfulness; relevancy pending)")

# Tabla resumen (promedios; recall excluye Q9, que es null, no 0).
print(f"{'variante':<10}{'precision':>11}{'recall':>9}{'faithfulness':>14}{'exactas':>9}")
for name in VARIANTS:
    pq = out["variants"][name]["per_question"]
    prec = [v["context_precision"] for v in pq.values() if v["context_precision"] is not None]
    rec = [v["context_recall"] for v in pq.values() if v["context_recall"] is not None]
    fai = [v["faithfulness"] for v in pq.values() if v["faithfulness"] is not None]
    n_exact = sum(1 for v in pq.values() if v["exactness_correct"])
    print(f"{name:<10}{sum(prec) / len(prec):>11.2f}{sum(rec) / len(rec):>9.2f}{sum(fai) / len(fai):>14.2f}{n_exact:>8}/{len(pq)}")

In [ ]:
# answer_relevancy (proxy): coseno entre embeddings all-MiniLM-L6-v2 de la
# pregunta y la respuesta. RAGAS la define via preguntas generadas por LLM a
# partir de la respuesta; ese paso NO se usa aqui: es un proxy, etiquetado
# como tal en metrics_ragas.json.
import json, time
from pathlib import Path
import numpy as np
from sentence_transformers import SentenceTransformer

BASE = Path(REPO) / "rag_baseline"
RAGAS_DIR = BASE / "ragas"
TRACES = {"dense": f"run_trace{SUF}.jsonl", "hybrid": f"run_trace_hybrid{SUF}.jsonl", "expanded": f"run_trace_expanded{SUF}.jsonl"}

t0 = time.time()
model = SentenceTransformer("all-MiniLM-L6-v2")
metrics = json.load(open(RAGAS_DIR / f"metrics_ragas{SUF}.json"))
for name, trace_file in TRACES.items():
    trace = [json.loads(l) for l in open(BASE / trace_file)]
    qs = [t["question"] for t in trace]
    ans = [t["final_answer"] for t in trace]
    eq = model.encode(qs, normalize_embeddings=True, show_progress_bar=False)
    ea = model.encode(ans, normalize_embeddings=True, show_progress_bar=False)
    for t, a, b in zip(trace, eq, ea):
        metrics["variants"][name]["per_question"][t["id"]]["answer_relevancy"] = round(float(np.dot(a, b)), 4)
    print(name, "done")
metrics["answer_relevancy_method"] = "proxy: cosine(all-MiniLM-L6-v2(question), all-MiniLM-L6-v2(answer)); RAGAS defines it via LLM-generated questions, not used here"
json.dump(metrics, open(RAGAS_DIR / f"metrics_ragas{SUF}.json", "w"), ensure_ascii=False, indent=1)
print("relevancy computed in %.1fs" % (time.time() - t0))

# Tabla resumen de relevancy (promedio por variante).
print(f"{'variante':<10}{'answer_relevancy':>17}")
for name in TRACES:
    rel = [v["answer_relevancy"] for v in metrics["variants"][name]["per_question"].values()]
    print(f"{name:<10}{sum(rel) / len(rel):>17.2f}")

## 2. Piloto: una muestra con el juez Ollama

Antes del run completo, una sola muestra (denso Q1, la interesante: el lector contestó 40,236 con 781,585 en contexto) con las 4 métricas y `RunConfig`. Si el piloto no completa en tiempo razonable, el run completo no es viable y nos quedamos con las deterministas — igual que pasó en CPU.

In [ ]:
import json, time
from pathlib import Path
from ragas import EvaluationDataset, SingleTurnSample, evaluate
from ragas.metrics import AnswerRelevancy, ContextPrecision, ContextRecall, Faithfulness
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
try:
    from ragas import RunConfig
except ImportError:
    from ragas.run_config import RunConfig
from langchain_ollama import ChatOllama
from langchain_huggingface import HuggingFaceEmbeddings

BASE = Path(REPO) / "rag_baseline"
RAGAS_DIR = BASE / "ragas"

judge = ChatOllama(model=JUDGE_OLLAMA_MODEL, base_url="http://localhost:11434",
                   temperature=0, num_predict=512)
llm = LangchainLLMWrapper(judge)
emb = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2",
                          model_kwargs={"device": "cuda"}))
print("juez Ollama listo:", JUDGE_OLLAMA_MODEL)

trace = [json.loads(l) for l in open(BASE / f"run_trace{SUF}.jsonl")][0]  # denso Q1
gt = json.load(open(RAGAS_DIR / f"ground_truth_evidence{SUF}.json"))["questions"][0]
contexts = [c for c in trace["context"].split("\n\n") if c.strip()]
sample = SingleTurnSample(
    user_input=trace["question"],
    retrieved_contexts=contexts,
    response=trace["final_answer"],
    reference=gt["question"] + " Esperado: Ps. 781,585 millones, +11.2%.",
)
t1 = time.time()
result = evaluate(
    EvaluationDataset(samples=[sample]),
    metrics=[ContextPrecision(), ContextRecall(), Faithfulness(), AnswerRelevancy()],
    llm=llm, embeddings=emb, run_config=RunConfig(timeout=600),
)
print(f"piloto: {time.time() - t1:.1f}s")
print(result.to_pandas().to_string())

## 3. Run real completo (acotado por tiempo)

3 × N_Q muestras (N_Q preguntas × 3 variantes), una llamada `evaluate()` por muestra con las 4 métricas, como en el piloto de CPU (`pilot_timeout_900.py`). Tope `TIME_BUDGET_MIN`: las muestras que no se alcanzan quedan como `skipped`, no como números. Al final se comparan los valores juzgados por el LLM con los deterministas, pregunta por pregunta.

In [ ]:
import json, time
from pathlib import Path
from ragas import EvaluationDataset, SingleTurnSample, evaluate
from ragas.metrics import AnswerRelevancy, ContextPrecision, ContextRecall, Faithfulness
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
try:
    from ragas import RunConfig
except ImportError:
    from ragas.run_config import RunConfig
from langchain_ollama import ChatOllama
from langchain_huggingface import HuggingFaceEmbeddings

BASE = Path(REPO) / "rag_baseline"
RAGAS_DIR = BASE / "ragas"
OUT = RAGAS_DIR / f"metrics_ragas_llm_colab{SUF}.json"

judge = ChatOllama(model=JUDGE_OLLAMA_MODEL, base_url="http://localhost:11434",
                   temperature=0, num_predict=512)
llm = LangchainLLMWrapper(judge)
emb = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2",
                          model_kwargs={"device": "cuda"}))
METRICS = [ContextPrecision(), ContextRecall(), Faithfulness(), AnswerRelevancy()]
METRIC_NAMES = ["context_precision", "context_recall", "faithfulness", "answer_relevancy"]
run_config = RunConfig(timeout=600)

eval_q = {q["id"]: q for q in json.load(open(Path(REPO) / "memo" / EVAL_FILE))["questions"]}
det = json.load(open(RAGAS_DIR / f"metrics_ragas{SUF}.json"))["variants"]
gt = {q["id"]: q for q in json.load(open(RAGAS_DIR / f"ground_truth_evidence{SUF}.json"))["questions"]}

VARIANTS = [("dense", f"run_trace{SUF}.jsonl"),
            ("hybrid", f"run_trace_hybrid{SUF}.jsonl"),
            ("expanded", f"run_trace_expanded{SUF}.jsonl")]
traces = {}
for name, fname in VARIANTS:
    traces[name] = {r["id"]: r for r in (json.loads(l) for l in open(BASE / fname))}

results, t_start = {}, time.time()
budget_s = TIME_BUDGET_MIN * 60
for name, _ in VARIANTS:
    for qid in sorted(eval_q):
        if time.time() - t_start > budget_s:
            results[f"{name}/{qid}"] = {"status": "skipped",
                                        "reason": f"tope de tiempo ({TIME_BUDGET_MIN} min)"}
            continue
        tr = traces[name][qid]
        contexts = [c for c in tr["context"].split("\n\n") if c.strip()]
        sample = SingleTurnSample(
            user_input=tr["question"], retrieved_contexts=contexts,
            response=tr["final_answer"],
            reference=gt[qid]["question"] + " Esperado: " + eval_q[qid]["expected"])
        t1 = time.time()
        try:
            res = evaluate(EvaluationDataset(samples=[sample]), metrics=METRICS,
                           llm=llm, embeddings=emb, run_config=run_config)
            row = res.to_pandas().iloc[0].to_dict()
            vals = {m: (None if row.get(m) is None else float(row[m]))
                    for m in METRIC_NAMES}
            results[f"{name}/{qid}"] = {"status": "ok",
                                        "seconds": round(time.time() - t1, 1),
                                        "llm_judged": vals,
                                        "deterministic": det[name]["per_question"][qid]}
        except Exception as e:
            results[f"{name}/{qid}"] = {"status": "error",
                                        "seconds": round(time.time() - t1, 1),
                                        "error": f"{type(e).__name__}: {e}"}
        print(f"{name}/{qid}: {results[f'{name}/{qid}']['status']} "
              f"({results[f'{name}/{qid}'].get('seconds', '?')}s)", flush=True)

done = sum(1 for v in results.values() if v["status"] == "ok")
print(f"\ncompletadas: {done}/{3 * len(eval_q)} en {(time.time() - t_start) / 60:.1f} min")
json.dump({"judge": f"ollama {JUDGE_OLLAMA_MODEL} (Colab T4)",
           "results": results}, open(OUT, "w"), ensure_ascii=False, indent=2)
print(f"-> {OUT}")

## 4. Comparación: juzgado por LLM vs determinista

Si el run real completó muestras, esta celda compara los valores del juez con los deterministas (misma pregunta, misma variante) y resume acuerdos y desacuerdos. Si no completó ninguna, lo dice y las deterministas quedan como las cifras citables.

In [ ]:
import json

OUT = REPO + f"/rag_baseline/ragas/metrics_ragas_llm_colab{SUF}.json"
N_TOTAL = 3 * len(json.load(open(REPO + "/memo/" + EVAL_FILE))["questions"])
try:
    data = json.load(open(OUT))["results"]
except FileNotFoundError:
    print("No hay run real (no se corrio la celda 3 o no completo nada).")
    print("Las cifras citables son las deterministas de metrics_ragas.json.")
    data = {}

ok = {k: v for k, v in data.items() if v.get("status") == "ok"}
print(f"muestras juzgadas por LLM: {len(ok)}/{N_TOTAL}")
for k, v in sorted(ok.items()):
    lj, dt = v["llm_judged"], v["deterministic"]
    diffs = " | ".join(
        f"{m}: LLM {lj[m] if lj[m] is not None else 'null'} vs det "
        f"{dt[m] if dt.get(m) is not None else 'null'}"
        for m in ["context_precision", "context_recall", "faithfulness", "answer_relevancy"])
    print(f"{k}: {diffs}")

## Guardar en Drive (final)

Todo lo que produjo este notebook vive bajo el repo clonado en Drive (`REPO`), así que sobrevive aunque la sesión de Colab se corte. Esta celda lo verifica y lo lista: si algo falta, lo dice aquí, no después.

Si el run real no completó muestras, `metrics_ragas_llm_colab{SUF}.json` no existe y la celda lo marca como pendiente: las deterministas siguen siendo las cifras citables.

In [ ]:
import os, shutil, time


def _human(n):
    for u in ["B", "KB", "MB", "GB"]:
        if n < 1024:
            return f"{n:.0f} {u}"
        n /= 1024
    return f"{n:.1f} GB"


def _list_dir(path):
    total, nfiles = 0, 0
    for root, _dirs, files in os.walk(path):
        for fn in sorted(files):
            p = os.path.join(root, fn)
            sz = os.path.getsize(p)
            total += sz
            nfiles += 1
            print(f"    {os.path.relpath(p, REPO)} ({_human(sz)})")
    return nfiles, total


# Logs de la sesion (viven en /content, fuera de Drive): copiarlos al repo.
for _log in ["ollama.log", "ollama_error.log", "relate.log"]:
    if os.path.isfile(_log):
        shutil.copy(_log, os.path.join(REPO, _log))
        print(f"log copiado a Drive: {_log}")

ENTRIES = [('rag_baseline/ragas/metrics_ragas{SUF}.json', 'Métricas RAGAS deterministas (actualizadas en Colab)', True), ('rag_baseline/ragas/metrics_ragas_llm_colab{SUF}.json', 'Run real ragas con juez Ollama (3x{N_Q} muestras)', False), ('rag_baseline/ragas/ground_truth_evidence{SUF}.json', 'Ground truth auditable', False)]
total_bytes, missing = 0, []
for rel, label, required in ENTRIES:
    # {SUF}/{N_Q} se resuelven con los parametros del notebook (EVAL_SET);
    # en notebooks sin esos parametros valen "" y 12.
    rel = rel.format(SUF=globals().get("SUF", ""), N_Q=globals().get("N_Q", 12))
    label = label.format(SUF=globals().get("SUF", ""), N_Q=globals().get("N_Q", 12))
    p = os.path.join(REPO, rel)
    if os.path.isdir(p):
        nf, tb = _list_dir(p)
        total_bytes += tb
        print(f"{label}: directorio, {nf} archivos, {_human(tb)}")
    elif os.path.isfile(p):
        sz = os.path.getsize(p)
        total_bytes += sz
        mt = time.strftime("%Y-%m-%d %H:%M", time.localtime(os.path.getmtime(p)))
        print(f"{label}: {rel} ({_human(sz)}, {mt})")
    else:
        status = "FALTA (obligatorio)" if required else "pendiente (opcional)"
        print(f"{label}: {status}: {rel}")
        if required:
            missing.append(rel)
print(f"\nGUARDADO EN DRIVE: {_human(total_bytes)} verificados bajo {REPO}")
if missing:
    print("OJO, faltan obligatorios:", missing)